# Data Conversion of SBE39plus .hex data
This python notebook shows how to convert a .hex file into pandas dataframe with scientific values.

Please contact [SBS customer support](https://www.seabird.com/support) for help or to request additional features.

## Init

The initialization code section below is used to import required libraries.

In [ ]:
# Native imports
from pathlib import Path

# relative path imports
import example_data.example_coefficients as ec

# Third-party imports
import numpy as np
import xarray as xr

# Sea-Bird imports
import seabirdscientific.conversion as sc
import seabirdscientific.instrument_data as si
import seabirdscientific.visualization as sv

## [Data Conversion](#proc-list)

This section shows how to convert raw data contained in a .hex file into scientific units for the instruments that follow:  
- SBE39plus
- SBE39plus IM

In [ ]:
hex_file = Path("example_data/SBE39plus/SBE39plus.hex")

# Convert raw hexadecimal string to raw frequencies
raw_data = si.read_hex_file(
    filepath=hex_file,
    instrument_type=si.InstrumentType.SBE39Plus,
    enabled_sensors=[
        si.Sensors.Temperature,
        si.Sensors.Pressure,
    ],
)

In [ ]:
# Convert raw frequencies to scientific values

temperature = sc.convert_temperature(
    temperature_counts_in=raw_data["temperature"].values,
    coefs=ec.temperature_coefs_sn03906502,
    standard="ITS90",
    units="C",
    use_mv_r=False,
)

pressure = sc.convert_pressure(
    pressure_count=raw_data["pressure"].values,
    compensation_voltage=raw_data["temperature compensation"].values,
    coefs=ec.pressure_coefs_sn03906502,
    units="dbar",
)

# Flag to be used in data processing
flag = np.zeros(len(temperature))

dataset = xr.Dataset(
    coords={"scan": np.arange(len(temperature))},
    data_vars={
        "temperature": ("scan", temperature),
        "pressure": ("scan", pressure),
        "date_time": ("scan", raw_data["date time"].values),
        "flag": ("scan", flag),
    },
    attrs={"file_name": "SBE39plus.hex"},
)

dataset

## [Data Plotting](#proc-list)

In [ ]:
config = sv.ChartConfig(
    title="SBE39plus Data Conversion",
    x_names=["date_time"],
    y_names=["temperature", "pressure"],
    z_names=[],
    chart_type="overlay",
    plot_loop_edit_flags=False,
    lift_pen_over_bad_data=True,
)

fig = sv.plot_xy_chart(dataset, config)

# plotly customizations
fig["layout"]["yaxis"]["autorange"] = "reversed"
fig.data[0].name = "Temperature"
fig.data[1].name = "Pressure"
fig["layout"]["yaxis"]["title"] = "Temperature [ITS-90 degrees C]"
fig["layout"]["yaxis2"]["title"] = "Pressure [dbar]"


fig.update_layout(height=800)
fig.show()

## Processing

This converted data should now be processed using the tools in processing.ipynb to produce a final data product.